In [1]:
# # 1. Gỡ cài đặt các thư viện cũ
# !pip uninstall numpy pandas prophet cmdstanpy -y

# # 2. Cài đặt đúng phiên bản (theo project gốc)
# !pip install numpy==1.26.4
# !pip install pandas==2.2.2
# !pip install lightgbm==4.3.0 optuna==3.6.1 scikit-learn==1.5.1
# !pip install matplotlib==3.6.0 seaborn==0.13.2 shap==0.44.1
#!pip install streamlit==1.35.0 joblib==1.4.2 pyarrow==20.0.0
!pip install cmdstanpy
!pip install prophet

In [2]:
import numpy as np
import pandas as pd
import prophet
import joblib
import os
import sys
import torch

if torch.cuda.is_available():
    print(f"GPU activated: {torch.cuda.get_device_name(0)}")
else:
    print(" still CPU")

print("NumPy version:", np.__version__)      # Phải là 1.26.4
print("Pandas version:", pd.__version__)     # Phải là 2.2.2
print("Prophet version:", prophet.__version__) # Phải là 1.1.4

In [3]:

# Thêm trực tiếp thư mục gốc của project vào sys.path
project_root = '/content/Walmart_sales_forecasting'
if project_root not in sys.path:
    sys.path.append(project_root)

print("Đã thêm đường dẫn:", project_root)

In [4]:
import pandas as pd
import numpy as np
import torch
import prophet
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from src.metrics import weighted_absolute_percentage_error


In [5]:
%cd /content/Walmart_sales_forecasting
data_dir = 'data/processed/feature_engineering.feather'
df_feature = pd.read_feather(data_dir)
df_feature

In [6]:
df_feature.info()

In [7]:
from sklearn.preprocessing import StandardScaler

reg_cols = ['Size', 'Temperature', 'Fuel_Price', 'total_markdown', 'avg_markdown', 'max_markdown']
scaler = StandardScaler()
df_feature[reg_cols] = scaler.fit_transform(df_feature[reg_cols])

In [8]:
test = df_feature[df_feature['is_test']]
train = df_feature[~df_feature['is_test']]

prophet_data = df_feature.groupby(['Date', 'store_dept']).agg(
    {
        'Weekly_Sales' : 'sum',
        'Size' : 'first',
        'Type_B': 'first',         
        'Type_C': 'first',
        'IsHoliday_True' : 'first',
        'Temperature': 'first',
        'Fuel_Price' : 'first',
        "total_markdown": "first",   
        "avg_markdown": "first",      
        "max_markdown": "first",
    }
).reset_index()

prophet_data

In [9]:
#handle missing case
regressor_cols = ['Type_B', 'Type_C', 'IsHoliday_True', 'Temperature', 'Fuel_Price', 'total_markdown']
prophet_data[regressor_cols] = prophet_data[regressor_cols].fillna(0)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def build_prophet_model(prophet_data,  test_data, checkpoint_dir="/content/drive/MyDrive/Walmart_checkpoints"):
    
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    min_test_time = test_data['Date'].min()
    max_test_time = test_data['Date'].max()
    
    prophet_model = {}
    prophet_predictions = {}
    prophet_metrics = pd.DataFrame(columns=['combo', 'mae', 'rmse', 'wape'])
    all_real = []
    all_predict = [] 
    
    
    for combination in prophet_data['store_dept'].unique():
        print(f'Build prophet model for {combination}')
        
        model_path = os.path.join(checkpoint_dir,f"{combination}_model.pkl")
        forecast_path = os.path.join(checkpoint_dir,f"{combination}_forecast.csv")
        
        if os.path.exists(model_path) and os.path.exists(forecast_path):
            print(f"Load existing {combo}")
            model = joblib.load(model_path)
            forecast = pd.read_csv(forecast_path, parse_dates=['ds'])
            prophet_model[combo] = model
            prophet_predictions[combo] = forecast
            # recalc metrics
            mae = mean_absolute_error(forecast['y'], forecast['yhat'])
            rmse = np.sqrt(mean_squared_error(forecast['y'], forecast['yhat']))
            wape = weighted_absolute_percentage_error(forecast['y'], forecast['yhat'])
            prophet_metrics.loc[len(prophet_metrics)] = [combo, mae, rmse, wape]
            all_real.extend(forecast['y'])
            all_predict.extend(forecast['yhat'])
            continue
        
        combo = prophet_data[prophet_data['store_dept'] == combination]
        combo = combo.rename(columns={'Date':'ds', 'Weekly_Sales' : 'y'}) #prophet require ds and y
        
        combo_train = combo[combo['ds'] < min_test_time]
        combo_test = combo[(combo['ds'] >= min_test_time) & (combo['ds'] <= max_test_time)]
        if combo_train.empty or combo_test.empty:
            print(f'Skip {combination} due to lack of data')
            continue
        
        model = Prophet(daily_seasonality=False,weekly_seasonality=True, yearly_seasonality=True, seasonality_mode='multiplicative', changepoint_prior_scale=0.5)
        
        for reg in ['Type_B', 'Type_C', 'IsHoliday_True', 'Temperature', 'Fuel_Price', 'total_markdown']:
            model.add_regressor(reg)
        
        try:
            model.fit(combo_train)
        except Exception as e:
            print(f'Fail to train {combination} : {e}')
            continue
        
        # test
        future = combo_test[['ds', 'Size', 'Type_B', 'Type_C' , 'IsHoliday_True', 'Temperature', 'Fuel_Price', 'total_markdown', 'avg_markdown', 'max_markdown']]
        forecast = model.predict(future)
        
        forecast = forecast[['ds', 'yhat', 'yhat_upper', 'yhat_lower']].merge(combo_test[['ds', 'y']], on=['ds'])
        
        joblib.dump(model, model_path)
        forecast.to_csv(forecast_path, index=False)
        print(f"Saved checkpoint for {combo}")
        
        prophet_model[combination] = model
        prophet_predictions[combination] = forecast
        
        # evaluate
        mae = mean_absolute_error(forecast['y'], forecast['yhat'])
        rmse = np.sqrt(mean_squared_error(forecast['y'], forecast['yhat']))
        
        wampe = weighted_absolute_percentage_error(forecast['y'], forecast['yhat']).item()
        
        prophet_metrics[len(prophet_metrics)] = [combo, mae, rmse, wampe]
        
        #overall evaluation
        all_real.extend(forecast['y'])
        all_predict.extend(forecast['yhat'])
        
    
    mean_mae = np.mean(prophet_metrics['mae'])
    mean_rmse = np.mean(prophet_metrics['rmse'])
    ovr_wampe = weighted_absolute_percentage_error(all_real, all_predict)
    
    return prophet_model, prophet_predictions, (mean_mae, mean_rmse, ovr_wampe)


prophet_model, prophet_predictions, (mean_mae, mean_rmse, ovr_wampe) = build_prophet_model(prophet_data,  test)
    
        
        
        
        
        
        
        
        
        
        
    
    

In [ ]:
print(
    f"Prophet Model Results:\nMAE: {mean_mae:.2f} | RMSE: {mean_rmse:.2f} | WAPE: {ovr_wampe:.2f}%"
)